<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/W3_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import torch

In [2]:
from torch.utils.data import Dataset, DataLoader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/PRISM/processed/TCS.NS_processed.csv")

In [5]:
df.head()

,Date,Close,High,Low,Open,Volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target
0,2019-04-02,1709.451782,1714.960003,1674.675708,1674.757900,3719663,0.023454,0.023183,1654.418750,1653.493787,1651.925543,0.010694,0.774869,2817120.55,0.003775,0.872854,0.000000
1,2019-04-03,1709.451782,1717.919746,1692.022699,1714.137874,2939886,0.000000,0.000000,1670.310425,1658.138818,1655.374359,0.010672,-0.209636,2756687.45,-0.005912,0.846227,-0.031164
2,2019-04-04,1656.177979,1709.780720,1650.340886,1708.506381,4397518,-0.031164,-0.031660,1677.972656,1657.456470,1656.459564,0.012978,0.495812,2854082.25,-0.003946,0.927317,0.016778
3,2019-04-05,1683.966064,1688.980929,1659.713271,1667.811217,3152103,0.016778,0.016639,1685.865063,1660.190063,1658.461456,0.013428,-0.283209,2879935.05,0.005859,0.938799,0.010960
4,2019-04-08,1702.422729,1705.916776,1671.140697,1692.762719,2194294,0.010960,0.010901,1692.294067,1665.542102,1660.823022,0.013548,-0.303863,2862655.55,-0.005267,0.903789,0.010021


In [6]:
df.shape

(1417, 17)

In [7]:
df["Date"]=pd.to_datetime(df["Date"])

In [8]:
df.info() #dtype of date is object by default iss liye upar wali line mai convert kiya

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1417 entries, 0 to 1416
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           1417 non-null   datetime64[ns]
 1   Close          1417 non-null   float64       
 2   High           1417 non-null   float64       
 3   Low            1417 non-null   float64       
 4   Open           1417 non-null   float64       
 5   Volume         1417 non-null   int64         
 6   return         1417 non-null   float64       
 7   log_return     1417 non-null   float64       
 8   MA5            1417 non-null   float64       
 9   MA10           1417 non-null   float64       
 10  MA20           1417 non-null   float64       
 11  Volatility     1417 non-null   float64       
 12  Volume_Change  1417 non-null   float64       
 13  Volume_MA20    1417 non-null   float64       
 14  market_return  1417 non-null   float64       
 15  beta           1417 n

In [9]:
features=['Close','Volume','return','log_return','beta','MA5','MA10','MA20','Volume_Change','Volatility','Volume_MA20']
X = df[features]
y = df["target"]

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()         #mean is near to 0 and variance is approx 1

In [11]:
X_scaled = scaler.fit_transform(X)

In [12]:
print(X_scaled.shape)

(1417, 11)


In [13]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=features
)

In [14]:
X_scaled.head()

,Close,Volume,return,log_return,beta,MA5,MA10,MA20,Volume_Change,Volatility,Volume_MA20
0,-1.533568,0.590317,1.484408,1.474714,0.574918,-1.608368,-1.606102,-1.602249,0.668958,-0.620668,0.078936
1,-1.533568,0.120571,-0.046637,-0.038976,0.479710,-1.585999,-1.599555,-1.597370,-0.378741,-0.624522,0.016751
2,-1.608468,0.998665,-2.081029,-2.106184,0.769657,-1.575213,-1.600517,-1.595835,0.371990,-0.223548,0.116969
3,-1.569399,0.248413,1.048654,1.047460,0.810712,-1.564104,-1.596664,-1.593003,-0.457035,-0.145321,0.143571
4,-1.543450,-0.328582,0.668844,0.672764,0.685531,-1.555055,-1.589120,-1.589662,-0.479016,-0.124378,0.125791


In [15]:
sequence_length = 30
X_sequences=[]
y_sequences=[]

In [16]:
for i in range(len(X_scaled) - sequence_length):
    X_sequences.append(
        X_scaled.iloc[i:i+sequence_length].values
    )

    y_sequences.append(
        y.iloc[i+sequence_length]
    )

In [17]:
X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)

print(X_sequences.shape)
print(y_sequences.shape)

(1387, 30, 11)
(1387,)


In [18]:
X_tensor = torch.tensor(
    X_sequences,
    dtype=torch.float32
)
y_tensor = torch.tensor(
    y_sequences,
    dtype=torch.float32
)

In [19]:
print(X_tensor.shape)
print(y_tensor.shape)

torch.Size([1387, 30, 11])
torch.Size([1387])


In [20]:
print(X_tensor.dtype)
print(y_tensor.dtype)

torch.float32
torch.float32


In [21]:
print(X_tensor[0].shape)

torch.Size([30, 11])


In [22]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [23]:
dataset = StockDataset(
    X_tensor,
    y_tensor
)

In [24]:
len(dataset)

1387

In [26]:
x,y = dataset[0]
print(x.shape)
print(y)

torch.Size([30, 11])
tensor(-0.0160)


In [27]:
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [29]:
for X_batch, y_batch in loader:
    print(X_batch.shape)
    print(y_batch.shape)
    break

torch.Size([32, 30, 11])
torch.Size([32])


In [30]:
train_size = int(
    0.8 * len(dataset)
)
val_size = len(dataset) - train_size

In [31]:
train_dataset = torch.utils.data.Subset(
    dataset,
    range(train_size)
)
val_dataset = torch.utils.data.Subset(
    dataset,
    range(train_size,len(dataset))
)

In [32]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [33]:
print(len(train_dataset))
print(len(val_dataset))

1109
278


In [34]:
for X_batch,y_batch in train_loader:
    print(X_batch.shape)
    print(y_batch.shape)
    break

torch.Size([32, 30, 11])
torch.Size([32])
